In [64]:
"""
pipeline.py — Production ModerationPipeline class.
Implements a three-layer content moderation guardrail pipeline.

Usage:
    from pipeline import ModerationPipeline, load_pipeline
    pipe = load_pipeline()
    result = pipe.predict("some comment text")
    # Returns: {"decision": "block"|"allow"|"review",
    #           "layer": "input_filter"|"model"|"human_review",
    #           "confidence": float,
    #           "category": str (only for input_filter blocks)}

Prerequisites:
    - Run part4.ipynb first  → saves /kaggle/working/part4_best_model/
    - Run part5.ipynb first  → saves /kaggle/working/calibrator.pkl
"""

import re
import pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Layer 1: Regex Blocklist ─────────────────────────────────────────────────

BLOCKLIST = {
    "direct_threat": [
        re.compile(r"\bi\s+(will|'ll|am\s+going\s+to|gonna|would)\s+(kill|murder|shoot|stab|hurt|destroy|end)\s+(you|u|him|her|them|your\s+\w+)\b", re.IGNORECASE),
        re.compile(r"\b(you're|you\s+are|ur)\s+going\s+to\s+(die|get\s+hurt|get\s+shot|get\s+killed)\b", re.IGNORECASE),
        re.compile(r"\bsomeone\s+should\s+(kill|shoot|stab|hurt|murder)\s+(you|him|her|them)\b", re.IGNORECASE),
        re.compile(r"\bi('ll|\s+will)\s+find\s+(you|where\s+you\s+live|your\s+address)\b", re.IGNORECASE),
        re.compile(r"\b(i\s+hope|hope)\s+you\s+(die|get\s+killed|suffer|rot)\b", re.IGNORECASE),
        re.compile(r"\b(?P<verb>kill|murder|shoot|stab|hurt)\s+you\b", re.IGNORECASE),
    ],
    "self_harm_directed": [
        re.compile(r"\byou\s+should\s+(kill|hang|cut|hurt)\s+yourself\b", re.IGNORECASE),
        re.compile(r"\bgo\s+kill\s+yourself\b", re.IGNORECASE),
        re.compile(r"\bnobody\s+(would\s+miss|will\s+miss|cares\s+if)\s+you\b", re.IGNORECASE),
        re.compile(r"\bdo\s+(everyone|the\s+world)\s+a\s+(favour|favor)\s+and\s+(disappear|die|kill\s+yourself)\b", re.IGNORECASE),
    ],
    "doxxing_stalking": [
        re.compile(r"\bi\s+know\s+where\s+you\s+live\b", re.IGNORECASE),
        re.compile(r"\bi('ll|\s+will|'m\s+gonna)\s+post\s+your\s+(address|info|details|number)\b", re.IGNORECASE),
        re.compile(r"\bi\s+found\s+your\s+(real\s+name|address|location|number)\b", re.IGNORECASE),
        re.compile(r"\beveryone\s+will\s+know\s+who\s+you\s+(really\s+are|are)\b", re.IGNORECASE),
    ],
    "dehumanization": [
        re.compile(r"\b(?:humans?|people|persons?)\b.{0,30}\bnot\s+(?:human|real\s+people|people)\b", re.IGNORECASE),
        re.compile(r"\b(?:they|those\s+people|(?:immigrants?|refugees?|muslims?|jews?|blacks?|whites?))\s+are\s+(?:animals?|parasites?|vermin|subhuman|not\s+human)\b", re.IGNORECASE),
        re.compile(r"\b(?:immigrants?|refugees?|muslims?|jews?|blacks?|whites?|(?:human|people|persons?))\s+should\s+be\s+exterminated\b", re.IGNORECASE),
        re.compile(r"\b(?:they|those\s+(?:people|humans?))\s+are\s+a\s+disease\b", re.IGNORECASE),
    ],
    "coordinated_harassment": [
        re.compile(r"\beveryone\s+report\s+(?:this\s+)?@?\w+\b", re.IGNORECASE),
        re.compile(r"\blet'?s?\s+(all\s+)?go\s+after\s+@?\w+\b", re.IGNORECASE),
        re.compile(r"\braid\s+(their\s+)?(profile|account|stream|page)\b", re.IGNORECASE),
        re.compile(r"\bmass\s+report\s+(this\s+)?(account|user|profile)\b", re.IGNORECASE),
        re.compile(r"\b(?:let'?s?|we\s+should)\s+(?=.{0,20}(?:spam|flood|target|harass))", re.IGNORECASE),
    ],
}


def input_filter(text: str):
    """
    Layer 1: Regex pre-filter.
    Returns a block decision dict if matched, else None.
    """
    for category, patterns in BLOCKLIST.items():
        for pattern in patterns:
            if pattern.search(text):
                return {
                    "decision"  : "block",
                    "layer"     : "input_filter",
                    "category"  : category,
                    "confidence": 1.0,
                }
    return None


# ── ModerationPipeline ───────────────────────────────────────────────────────

class ModerationPipeline:
    """
    Three-layer production content moderation pipeline.

    Layer 1 — Input filter (regex):
        Catches high-confidence harm patterns instantly with no model call.
        Returns immediately on match.

    Layer 2 — Calibrated DistilBERT model:
        Runs isotonic-calibrated toxicity classifier.
        calibrated_prob >= 0.6  →  block
        calibrated_prob <= 0.4  →  allow
        0.4 < prob < 0.6        →  escalate to Layer 3

    Layer 3 — Human review queue:
        Uncertain cases routed to human reviewers.
        Returns decision='review' with confidence score.

    Parameters
    ----------
    model_dir   : Path to HuggingFace model directory (saved by Part 4).
    calibrator  : Fitted sklearn IsotonicRegression calibrator object.
    device      : torch.device (auto-detected if None).
    max_length  : Tokenizer max sequence length (default 128).
    """

    def __init__(self, model_dir: str, calibrator, device=None, max_length: int = 128):
        self.device     = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.calibrator = calibrator

        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.to(self.device)
        self.model.eval()

    def _raw_prob(self, text: str) -> float:
        """Get raw softmax probability of toxic class for a single text."""
        enc = self.tokenizer(
            [text],
            max_length     = self.max_length,
            truncation     = True,
            padding        = True,
            return_tensors = "pt",
        ).to(self.device)
        with torch.no_grad():
            logits = self.model(**enc).logits
        return torch.softmax(logits, dim=-1)[0, 1].item()

    def predict(self, text: str) -> dict:
        """
        Run the full three-layer pipeline on a single comment.

        Returns
        -------
        dict with keys:
            decision   : 'block' | 'allow' | 'review'
            layer      : 'input_filter' | 'model' | 'human_review'
            confidence : float  (1.0 for regex blocks; calibrated prob otherwise)
            category   : str    (only present when layer == 'input_filter')
        """
        # ── Layer 1 ──
        filter_result = input_filter(text)
        if filter_result is not None:
            return filter_result

        # ── Layer 2 ──
        raw_prob = self._raw_prob(text)
        cal_prob = float(self.calibrator.transform([raw_prob])[0])

        if cal_prob >= 0.6:
            return {"decision": "block",  "layer": "model",        "confidence": cal_prob}
        elif cal_prob <= 0.4:
            return {"decision": "allow",  "layer": "model",        "confidence": cal_prob}

        # ── Layer 3 ──
        return {"decision": "review", "layer": "human_review", "confidence": cal_prob}


# ── Convenience loader ───────────────────────────────────────────────────────

def load_pipeline(
    model_dir       : str = "/kaggle/working/part4_best_model",
    calibrator_path : str = "/kaggle/working/calibrator.pkl",
    device          = None,
    max_length      : int = 128,
) -> ModerationPipeline:
    """
    Load a saved ModerationPipeline from disk.
    Requires part4.ipynb and part5.ipynb to have been run first.
    """
    with open(calibrator_path, "rb") as f:
        calibrator = pickle.load(f)

    return ModerationPipeline(
        model_dir  = model_dir,
        calibrator = calibrator,
        device     = device,
        max_length = max_length,
    )


# ── Self-test ────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import os

    CAL_PATH = "/kaggle/working/calibrator.pkl"
    MDL_PATH = "/kaggle/working/part4_best_model"

    if not os.path.exists(CAL_PATH):
        print(f"ERROR: Calibrator not found at {CAL_PATH}. Run part5.ipynb first.")
    elif not os.path.exists(MDL_PATH):
        print(f"ERROR: Model not found at {MDL_PATH}. Run part4.ipynb first.")
    else:
        print("Loading pipeline …")
        pipe = load_pipeline(MDL_PATH, CAL_PATH)
        tests = [
            "I will kill you if you show up",
            "go kill yourself",
            "I know where you live and I'll post your address",
            "Have a wonderful day, hope you're doing well!",
            "This policy is absolutely stupid and wrong",
        ]
        print("\n=== pipeline.py Self-test ===")
        for t in tests:
            r = pipe.predict(t)
            print(f"  {r['decision'].upper():6s} | layer={r['layer']:14s} | conf={r['confidence']:.3f} | {t}")

Loading pipeline …


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== pipeline.py Self-test ===
  BLOCK  | layer=input_filter   | conf=1.000 | I will kill you if you show up
  BLOCK  | layer=input_filter   | conf=1.000 | go kill yourself
  BLOCK  | layer=input_filter   | conf=1.000 | I know where you live and I'll post your address
  ALLOW  | layer=model          | conf=0.082 | Have a wonderful day, hope you're doing well!
  ALLOW  | layer=model          | conf=0.069 | This policy is absolutely stupid and wrong
